In [1]:
!pip install tabpfn torch

  Using cached einops-0.8.1-py3-none-any.whl.metadata (13 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/241.3 MB ? eta -:--:--
    --------------------------------------- 5.5/241.3 MB 30.5 MB/s eta 0:00:08
   - -------------------------------------- 11.5/241.3 MB 30.1 MB/s eta 0:00:08
   --- ------------------------------------ 18.4/241.3 MB 30.5 MB/s eta 0:00:08
   --- ------------------------------------ 23.3/241.3 MB 28.9 MB/s eta 0:00:08
   ---- ----------------------------------- 28.0/241.3 MB 27.8 MB/s eta 0:00:08
   ----- ---------------------------------- 33.0/241.3 MB 26.9 MB/s eta 0:00:08
   ------ --------------------------------- 37.7/241.3 MB 26.1 MB/s eta 0:00:08
   ------- -------------------------------- 43.0/241.3 MB 26.0 MB/s eta 0:00:08
   ------- -------------------------------- 48.2/241.3 MB 26.0 MB/s eta 0:00:08
   -------- ------------------------------- 53.7/241.3 MB 25.9 MB/s eta 0:00:08
   --

In [5]:
import sys
print(sys.executable)


D:\Anaconda\envs\DL\python.exe


In [6]:
!python -m pip show tabpfn

Name: tabpfn
Version: 2.1.2
Summary: TabPFN: Foundation model for tabular data
Home-page: 


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\17609\.conda\envs\notebook\Lib\site-packages\pip\_internal\utils\logging.py", line 184, in emit
    self.console.print(renderable, overflow="ignore", crop=False, style=style)
  File "C:\Users\17609\.conda\envs\notebook\Lib\site-packages\pip\_vendor\rich\console.py", line 1678, in print
    with self:
         ^^^^
  File "C:\Users\17609\.conda\envs\notebook\Lib\site-packages\pip\_vendor\rich\console.py", line 864, in __exit__
    self._exit_buffer()
  File "C:\Users\17609\.conda\envs\notebook\Lib\site-packages\pip\_vendor\rich\console.py", line 822, in _exit_buffer
    self._check_buffer()
  File "C:\Users\17609\.conda\envs\notebook\Lib\site-packages\pip\_vendor\rich\console.py", line 2019, in _check_buffer
    self._write_buffer()
  File "C:\Users\17609\.conda\envs\notebook\Lib\site-packages\pip\_vendor\rich\console.py", line 2055, in _write_buffer
    legacy_windows_render(buffer, LegacyWindowsTerm(self.file))


In [7]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "show", "tabpfn"])


CalledProcessError: Command '['D:\\Anaconda\\envs\\DL\\python.exe', '-m', 'pip', 'show', 'tabpfn']' returned non-zero exit status 1.

In [8]:
import sys, subprocess
print("当前解释器：", sys.executable)  # 应该是 D:\Anaconda\envs\DL\python.exe

# 用“当前解释器”安装，避免装到别的环境
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "torch", "tabpfn"])

# 验证
subprocess.check_call([sys.executable, "-m", "pip", "show", "tabpfn"])
subprocess.check_call([sys.executable, "-m", "pip", "show", "torch"])


当前解释器： D:\Anaconda\envs\DL\python.exe


0

In [9]:
import sys, subprocess, importlib.util, site, platform

print("当前解释器:", sys.executable)
print("Python 版本:", platform.python_version())

def pip_run(args):
    print(">>> pip", " ".join(args))
    subprocess.check_call([sys.executable, "-m", "pip", *args])

# 1) 升级安装工具
pip_run(["install", "-U", "pip", "setuptools", "wheel"])

# 2) 安装/升级 torch（无GPU也可，自动CPU版；若你有CUDA想指定版本，再告诉我）
pip_run(["install", "-U", "torch"])

# 3) 安装/升级 tabpfn
pip_run(["install", "-U", "tabpfn"])

# 4) 验证：是否能被当前解释器发现
print("tabpfn spec:", importlib.util.find_spec("tabpfn"))
print("torch spec:", importlib.util.find_spec("torch"))

# 5) 试着导入并打印版本
import tabpfn, torch
print("tabpfn 版本:", getattr(tabpfn, "__version__", "unknown"))
print("torch 版本:", torch.__version__)


当前解释器: D:\Anaconda\envs\DL\python.exe
Python 版本: 3.9.21
>>> pip install -U pip setuptools wheel
>>> pip install -U torch
>>> pip install -U tabpfn
tabpfn spec: ModuleSpec(name='tabpfn', loader=<_frozen_importlib_external.SourceFileLoader object at 0x000002836C9E5E20>, origin='C:\\Users\\17609\\AppData\\Roaming\\Python\\Python39\\site-packages\\tabpfn\\__init__.py', submodule_search_locations=['C:\\Users\\17609\\AppData\\Roaming\\Python\\Python39\\site-packages\\tabpfn'])
torch spec: ModuleSpec(name='torch', loader=<_frozen_importlib_external.SourceFileLoader object at 0x000002836C9E5E50>, origin='C:\\Users\\17609\\AppData\\Roaming\\Python\\Python39\\site-packages\\torch\\__init__.py', submodule_search_locations=['C:\\Users\\17609\\AppData\\Roaming\\Python\\Python39\\site-packages\\torch'])
tabpfn 版本: 2.1.2
torch 版本: 2.8.0+cpu


In [10]:
from tabpfn import TabPFNRegressor

# 简单验证 TabPFN 是否可用
model = TabPFNRegressor(device="cpu")
print("TabPFN 模型创建成功！")


TabPFN 模型创建成功！


In [13]:
# ================================================================
# TabPFN 回归（自动适配版本的构造参数）
# 读取 -> 预处理 -> 划分 -> 训练 -> 评估 -> 导出（含可选置信区间）
# ================================================================
import os
import sys
import random
import warnings
import inspect
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 固定随机种子
os.environ['PYTHONHASHSEED'] = str(1)
np.random.seed(1)
random.seed(1)
warnings.filterwarnings("ignore")

# -------------------- 可选：是否输出置信区间 --------------------
USE_CONFORMAL = True   # True -> 开启 split conformal 区间；False -> 仅点预测
ALPHA = 0.10           # 1-ALPHA 的覆盖率（0.10 -> 90% 区间）

# -------------------- 设备自动选择（有 GPU 用 GPU） ---------------
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

# -------------------- 导入 TabPFN -------------------------------
try:
    from tabpfn import TabPFNRegressor
    import tabpfn as _tabpfn_mod
    _TABPFN_VER = getattr(_tabpfn_mod, "__version__", "unknown")
except Exception as e:
    raise ImportError("未找到 TabPFN，请先在当前解释器安装：python -m pip install -U tabpfn torch") from e

def build_tabpfn(device: str = DEVICE):
    """
    构造 TabPFNRegressor，自动探测支持的参数，适配不同版本：
    - 老版本：存在 N_ensemble_configurations
    - 新版本：存在 batch_size / max_epochs
    - 中版本：仅支持 device
    """
    sig = inspect.signature(TabPFNRegressor.__init__)
    params = sig.parameters

    if "N_ensemble_configurations" in params:
        # 老版本 API
        return TabPFNRegressor(
            N_ensemble_configurations=32,
            seed=1,
            device=device
        )
    elif "batch_size" in params:
        # 新版 API（支持 batch_size / max_epochs）
        return TabPFNRegressor(
            device=device,
            batch_size=128,
            max_epochs=40
        )
    elif "device" in params:
        # 中版 API（仅 device）
        return TabPFNRegressor(device=device)
    else:
        # 极旧兜底
        return TabPFNRegressor()

# -------------------- 1) 读取数据 -------------------------------
data = pd.read_excel('dataset.xlsx')   # 确保文件在当前工作目录
# 若需要把 "C0" 改名为带下标的列名可保留，若无该列名不会报错
if "C0" in data.columns:
    data.rename(columns={"C0": r"C$_0$"}, inplace=True)

# 约定：最后一列为目标 y，其余为特征 X
X = data.iloc[:, :-1].copy()
y = data.iloc[:, -1].copy()

# 转为数值并检查缺失
X = X.apply(pd.to_numeric, errors='coerce')
y = pd.to_numeric(y, errors='coerce')
if X.isnull().any().any() or y.isnull().any():
    raise ValueError("数据中存在 NaN/Inf，请先清洗。")

# -------------------- 2) 预处理与划分 ----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.3, random_state=1
)

# -------------------- 3) 训练（含可选 Conformal） -----------------
def fit_and_predict_with_optional_conformal(X_train, y_train, X_test,
                                            use_conformal=USE_CONFORMAL, alpha=ALPHA):
    """
    - use_conformal=True：训练集再切一部分做校准，返回点预测 + (L, U) 区间
    - use_conformal=False：仅返回点预测，区间为 None
    """
    if use_conformal:
        # 训练-校准划分（split conformal）
        X_fit, X_cal, y_fit, y_cal = train_test_split(
            X_train, y_train, test_size=0.2, random_state=1
        )
        model = build_tabpfn(DEVICE)
        model.fit(X_fit, y_fit)

        # 校准残差：|y - y_hat|
        y_cal_pred = model.predict(X_cal)
        cal_res = np.abs(np.asarray(y_cal) - np.asarray(y_cal_pred))

        # 分位数阈值（常用近似 q = quantile(residuals, 1 - alpha)）
        q = np.quantile(cal_res, 1 - alpha)

        # 训练/测试集预测
        y_pred_train = model.predict(X_train)
        y_pred_test  = model.predict(X_test)

        # 置信区间
        L_train = y_pred_train - q
        U_train = y_pred_train + q
        L_test  = y_pred_test  - q
        U_test  = y_pred_test  + q

        intervals = {
            "train": (L_train, U_train),
            "test":  (L_test,  U_test),
            "q": q
        }
        return model, y_pred_train, y_pred_test, intervals
    else:
        model = build_tabpfn(DEVICE)
        model.fit(X_train, y_train)
        y_pred_train = model.predict(X_train)
        y_pred_test  = model.predict(X_test)
        return model, y_pred_train, y_pred_test, None

model, y_pred_train, y_pred_test, intervals = fit_and_predict_with_optional_conformal(
    X_train, y_train, X_test, use_conformal=USE_CONFORMAL, alpha=ALPHA
)

# -------------------- 4) 评估 -------------------------------
def evaluate(y_train, y_pred_train, y_test, y_pred_test):
    metrics = {
        "Train RMSE": float(np.sqrt(mean_squared_error(y_train, y_pred_train))),
        "Test RMSE":  float(np.sqrt(mean_squared_error(y_test,  y_pred_test))),
        "Train R^2":  float(r2_score(y_train, y_pred_train)),
        "Test R^2":   float(r2_score(y_test,  y_pred_test)),
        "Train MAE":  float(mean_absolute_error(y_train, y_pred_train)),
        "Test MAE":   float(mean_absolute_error(y_test,  y_pred_test)),
    }
    return metrics

metrics = evaluate(y_train, y_pred_train, y_test, y_pred_test)

print("=== TabPFN 回归结果（版本 {}，设备 {}）===".format(_TABPFN_VER, DEVICE))
for k, v in metrics.items():
    print(f"{k}: {v:.6f}")

if intervals is not None:
    print(f"[Conformal] 目标覆盖率: {int((1-ALPHA)*100)}% ，校准阈值 q = {intervals['q']:.6f}")

# -------------------- 5) 导出到 Excel -------------------------
def export_results(y_train, y_pred_train, y_test, y_pred_test,
                   intervals=None, filename="Results_TabPFN.xlsx"):
    with pd.ExcelWriter(filename) as writer:
        train_df = pd.DataFrame({
            "y_true": np.asarray(y_train),
            "y_pred": np.asarray(y_pred_train)
        })
        test_df = pd.DataFrame({
            "y_true": np.asarray(y_test),
            "y_pred": np.asarray(y_pred_test)
        })
        if intervals is not None:
            L_tr, U_tr = intervals["train"]
            L_te, U_te = intervals["test"]
            train_df["PI_Lower"] = np.asarray(L_tr)
            train_df["PI_Upper"] = np.asarray(U_tr)
            test_df["PI_Lower"]  = np.asarray(L_te)
            test_df["PI_Upper"]  = np.asarray(U_te)

        train_df.to_excel(writer, sheet_name="Train Results", index=False)
        test_df.to_excel(writer, sheet_name="Test Results", index=False)

export_results(y_train, y_pred_train, y_test, y_pred_test, intervals=intervals)

print("导出完成：Results_TabPFN.xlsx")


=== TabPFN 回归结果（版本 2.1.2，设备 cpu）===
Train RMSE: 10.660819
Test RMSE: 4.069798
Train R^2: 0.926234
Test R^2: 0.990325
Train MAE: 5.097336
Test MAE: 2.019198
[Conformal] 目标覆盖率: 90% ，校准阈值 q = 19.073518
导出完成：Results_TabPFN.xlsx
